# Lab 04 — Source Preparation

This notebook prepares the **UCI Online Retail** workbook for repeatable Bronze, Silver, schema-evolution, data-quality, and SCD demonstrations.

## Objectives

By the end of this notebook, you will have:

- verified that the uploaded workbook exists and follows the expected eight-column source contract;
- profiled row counts, duplicate rows, data types, and nulls before transformation;
- converted Excel into a Spark-friendly Parquet representation while preserving the original workbook;
- added technical lineage metadata without applying Silver business rules;
- created deterministic initial, incremental, evolved, invalid, schema-mismatch, and SCD-change batches;
- validated that the generated batches are complete and reproducible.

> **Scope boundary:** this notebook prepares source scenarios. Cleaning, quarantine, deduplication, MERGE, and SCD logic belong to later notebooks.

## 1. Install the Excel reader dependency

Databricks includes pandas, Spark, and PyArrow, but `openpyxl` is not guaranteed to be installed in every runtime. This bootstrap cell installs it only when it is missing, so **Run all** works without a separate manual library step.

In [0]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("openpyxl") is None:
    print("Installing openpyxl for Excel workbook support...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "openpyxl>=3.1,<4",
    ])

import openpyxl
print(f"openpyxl ready: {openpyxl.__version__}")

Installing openpyxl for Excel workbook support...
Looking in indexes: [REDACTED]
openpyxl ready: 3.1.5


## 2. Load the shared Lab 4 configuration

The configuration notebook creates the Unity Catalog volume and exposes all shared paths, table names, widgets, and contract settings. Running it here prevents path and naming differences between notebooks.

In [0]:
%run ./lab04_00_config

# Lab 04 — Configuration and Unity Catalog Setup

This notebook:
- defines Lab 4 parameters;
- creates the Unity Catalog catalog and schema when permitted;
- creates a managed or external volume;
- builds source, staging, landing, schema, checkpoint, quarantine, and test folders;
- defines all Bronze, Silver, SCD, and demonstration table names.

Run this notebook first. It is a setup notebook and does not need to be scheduled in the production Job.

Catalog ready: dbr_dev
Schema ready: dbr_dev.parvinbadalov
Volume ready: dbr_dev.parvinbadalov.lab04_silver_quality (external)
External location: abfss://parvinbadalov@dlspl21databricks.dfs.core.windows.net/lab04_silver_quality


Created or verified 13 Lab 4 folders under /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
  source: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source
  staging_initial: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial
  staging_incremental: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/incremental
  staging_evolved: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/evolved
  staging_invalid: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/invalid
  landing: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing
  schema_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/schema/bronze
  checkpoint_bronze: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/bronze
  checkpoint_silver: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/checkpoints/silver
  quarantine: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine
  schema_mismatch: /Volumes/dbr_dev/parvinba

Lab 4 table names configured:
  bronze: dbr_dev.parvinbadalov.lab04_bronze_retail
  silver_transactions: dbr_dev.parvinbadalov.lab04_silver_transactions
  quarantine: dbr_dev.parvinbadalov.lab04_quarantine
  quality_metrics: dbr_dev.parvinbadalov.lab04_quality_metrics
  product_scd0: dbr_dev.parvinbadalov.lab04_product_scd0
  product_scd1: dbr_dev.parvinbadalov.lab04_product_scd1
  product_scd2: dbr_dev.parvinbadalov.lab04_product_scd2
  product_scd3: dbr_dev.parvinbadalov.lab04_product_scd3
  product_scd4_current: dbr_dev.parvinbadalov.lab04_product_current
  product_scd4_history: dbr_dev.parvinbadalov.lab04_product_history
  product_scd6: dbr_dev.parvinbadalov.lab04_product_scd6
  schema_demo: dbr_dev.parvinbadalov.lab04_schema_demo
  column_mapping_demo: dbr_dev.parvinbadalov.lab04_column_mapping_demo


Upload the workbook to: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Expected columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Active contract: v1
Schema policy: fail
Trigger: availableNow; maximum files per trigger: 50


Volume validation succeeded; 6 top-level entries found.
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/quarantine/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/system/
dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/


In [0]:
from pathlib import Path
import zipfile

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

print(f"Source workbook: {source_file_path}")
print(f"Active contract: {contract_version}")
print(f"Schema policy for later tests: {schema_policy}")

Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Active contract: v1
Schema policy for later tests: fail


## 3. Verify the uploaded workbook

A filename ending in `.xlsx` is not sufficient evidence that a valid workbook was uploaded. Modern Excel files are ZIP-based packages, so this section verifies the file exists, is non-empty, and has a valid workbook structure before an expensive read begins.

If this check fails, upload a fresh copy of `Online Retail.xlsx` to the exact path printed by the configuration notebook.

In [0]:
source_files = dbutils.fs.ls(source_path)
matching_files = [item for item in source_files if item.name == source_file_name]

if not matching_files:
    raise FileNotFoundError(
        f"Upload {source_file_name!r} to {source_file_path} before continuing."
    )

source_file_info = matching_files[0]
if source_file_info.size <= 0:
    raise ValueError(f"The uploaded workbook is empty: {source_file_path}")

if not zipfile.is_zipfile(source_file_path):
    raise ValueError(
        "The uploaded file is not a valid .xlsx package. "
        "Download a fresh workbook from the UCI source and upload it again."
    )

print("Workbook file validation succeeded.")
print(f"Name: {source_file_info.name}")
print(f"Size: {source_file_info.size:,} bytes")
print(f"Path: {source_file_info.path}")

Workbook file validation succeeded.
Name: Online Retail.xlsx
Size: 23,715,344 bytes
Path: dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx


## 4. Read Excel and validate the source contract

Spark does not read `.xlsx` natively. Pandas with `openpyxl` is used only at this boundary. The data is then converted to Parquet so all later processing uses Spark and Delta-compatible formats.

The contract check is intentionally strict: all eight required columns must exist. Additional columns are reported because uncontrolled source additions should be reviewed rather than silently accepted.

In [0]:
try:
    workbook = pd.ExcelFile(source_file_path, engine="openpyxl")
except ImportError as exc:
    raise ImportError(
        "openpyxl could not be imported after dependency bootstrap. "
        "Rerun this notebook from the first cell."
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"The workbook could not be opened: {source_file_path}. "
        "Confirm that it is a complete, valid .xlsx file."
    ) from exc

source_sheet = "Online Retail" if "Online Retail" in workbook.sheet_names else workbook.sheet_names[0]
print(f"Worksheets: {workbook.sheet_names}")
print(f"Selected worksheet: {source_sheet}")

source_pdf = pd.read_excel(
    workbook,
    sheet_name=source_sheet,
    engine="openpyxl",
)
source_pdf.columns = [str(column).strip() for column in source_pdf.columns]

actual_columns = list(source_pdf.columns)
missing_columns = [column for column in expected_source_columns if column not in actual_columns]
additional_columns = [column for column in actual_columns if column not in expected_source_columns]

if missing_columns:
    raise ValueError(f"Source contract failed. Missing columns: {missing_columns}")

print("Required-column validation succeeded.")
print(f"Required columns: {expected_source_columns}")
print(f"Additional source columns: {additional_columns or 'none'}")

Worksheets: ['Online Retail']
Selected worksheet: Online Retail
Required-column validation succeeded.
Required columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Additional source columns: none


## 5. Profile the raw workbook

Profiling happens before any type normalization so the evidence reflects what arrived from the source. The Online Retail dataset contains real-world complications such as missing customer IDs, cancellations, negative quantities, and duplicate rows. These are observations—not all are automatically errors.

In particular, an invoice beginning with `C` or a negative quantity usually represents a cancellation and will be handled explicitly in the Silver layer.

In [0]:
raw_row_count = len(source_pdf)
raw_duplicate_count = int(source_pdf.duplicated().sum())
raw_null_counts = source_pdf[expected_source_columns].isna().sum().astype(int)

print(f"Rows: {raw_row_count:,}")
print(f"Columns: {len(actual_columns)}")
print(f"Exact duplicate rows: {raw_duplicate_count:,}")
print("Raw pandas data types:")
print(source_pdf[expected_source_columns].dtypes.to_string())

null_profile_pdf = (
    raw_null_counts
    .rename_axis("column_name")
    .reset_index(name="null_count")
)
null_profile_pdf["null_percentage"] = (
    null_profile_pdf["null_count"] / max(raw_row_count, 1) * 100
).round(4)

display(null_profile_pdf)

# Databricks display() converts pandas data through Arrow. The raw workbook
# intentionally contains mixed Excel values (for example numeric and
# alphanumeric StockCode entries), so render this diagnostic sample as HTML.
# This affects only the preview; source_pdf remains unchanged.
sample_preview_pdf = source_pdf[expected_source_columns].head(20)
displayHTML(sample_preview_pdf.to_html(index=False, na_rep="NULL"))

Rows: 541,909
Columns: 8
Exact duplicate rows: 5,268
Raw pandas data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object


column_name,null_count,null_percentage
InvoiceNo,0,0.0
StockCode,0,0.0
Description,1454,0.2683
Quantity,0,0.0
InvoiceDate,0,0.0
UnitPrice,0,0.0
CustomerID,135080,24.9267
Country,0,0.0


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


## 6. Normalize transport types and convert to Parquet

This is **technical normalization**, not Silver cleaning. It gives Spark predictable transport types while retaining questionable business values for later quality tests. For example, negative quantities and missing customer IDs are preserved.

The original Excel workbook remains unchanged. A separate Parquet copy is created under `source/prepared/`, making reruns faster and avoiding repeated Excel parsing. Timestamp values are written with microsecond precision because Spark does not support Parquet `TIMESTAMP(NANOS)`.

In [0]:
def normalize_identifier(value):
    if pd.isna(value):
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


prepared_pdf = source_pdf[expected_source_columns].copy()
prepared_pdf["InvoiceNo"] = prepared_pdf["InvoiceNo"].map(normalize_identifier)
prepared_pdf["StockCode"] = prepared_pdf["StockCode"].map(normalize_identifier)
prepared_pdf["Description"] = prepared_pdf["Description"].map(normalize_identifier)
prepared_pdf["CustomerID"] = prepared_pdf["CustomerID"].map(normalize_identifier)
prepared_pdf["Country"] = prepared_pdf["Country"].map(normalize_identifier)
prepared_pdf["Quantity"] = pd.to_numeric(prepared_pdf["Quantity"], errors="coerce").astype("Int64")
prepared_pdf["InvoiceDate"] = (
    pd.to_datetime(prepared_pdf["InvoiceDate"], errors="coerce")
    .astype("datetime64[us]")
)
prepared_pdf["UnitPrice"] = pd.to_numeric(prepared_pdf["UnitPrice"], errors="coerce").astype("float64")
prepared_pdf["_source_row_number"] = range(2, len(prepared_pdf) + 2)

prepared_source_dir = f"{source_path}/prepared"
prepared_source_file = f"{prepared_source_dir}/online_retail_source.parquet"
dbutils.fs.mkdirs(prepared_source_dir)

prepared_pdf.to_parquet(
    prepared_source_file,
    engine="pyarrow",
    index=False,
    coerce_timestamps="us",
    allow_truncated_timestamps=True,
)

print("Excel-to-Parquet conversion completed.")
print(f"Prepared source: {prepared_source_file}")

Excel-to-Parquet conversion completed.
Prepared source: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/prepared/online_retail_source.parquet


## 7. Add technical lineage metadata

Technical metadata makes each record traceable without changing its business meaning:

- `_source_file` and `_source_sheet` identify the origin;
- `_source_row_number` locates the row in the original worksheet;
- `_prepared_at_utc` records when this preparation run occurred;
- `_record_hash` is a deterministic hash of the eight source fields and supports later duplicate detection.

The hash deliberately excludes preparation metadata, so it remains stable across reruns.

In [0]:
source_spark_df = spark.read.parquet(prepared_source_file)

hash_expression = F.concat_ws(
    "||",
    *[F.coalesce(F.col(column).cast("string"), F.lit("<NULL>")) for column in expected_source_columns],
)

prepared_df = (
    source_spark_df
    .withColumn("_source_file", F.lit(source_file_name))
    .withColumn("_source_sheet", F.lit(source_sheet))
    .withColumn("_prepared_at_utc", F.current_timestamp())
    .withColumn("_record_hash", F.sha2(hash_expression, 256))
)

prepared_row_count = prepared_df.count()
if prepared_row_count != raw_row_count:
    raise AssertionError(
        f"Row-count mismatch after conversion: raw={raw_row_count}, prepared={prepared_row_count}"
    )

prepared_df.printSchema()
print(f"Prepared rows: {prepared_row_count:,}")
display(prepared_df.limit(20))

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_row_number: long (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _source_sheet: string (nullable = false)
 |-- _prepared_at_utc: timestamp (nullable = false)
 |-- _record_hash: string (nullable = true)

Prepared rows: 541,909


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_source_row_number,_source_file,_source_sheet,_prepared_at_utc,_record_hash
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00.000,2.55,17850,United Kingdom,2,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,b7cdeaae1b429903c2c3806825910d57f385aae6bcfdaa1d123fd2ef78a659ae
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,3,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,532942c37207951778ca5258c9b4de0cf4b789f947acdff1ebb7747857250548
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000,2.75,17850,United Kingdom,4,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,c1cd806692dd9c0f64a38acc337200ebc4eff66727b38ab69269e4a98bd4744f
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,5,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,97792215273e2a88a600b1e0c97a2cc746f3dd545256ad9a76ab05e045495e91
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,6,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,5d2f7ca64405415988e613920f8bbd7b3dd24e7632def92f5a560dd284d8d55b
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000,7.65,17850,United Kingdom,7,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,0d605467e0138f6396f9c149ce5ad5ab7096c7056518578056ef5dc2dc38dfbd
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000,4.25,17850,United Kingdom,8,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,d437e01828afac453bca294ee923cdc7ae1985b362293d1004075e5b0cc14b40
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000,1.85,17850,United Kingdom,9,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,67dbee6035f7cbad53ae0c05b2431427c7772468c33e923cb062a30ad382cfa4
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000,1.85,17850,United Kingdom,10,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,746abb26f54dbc3cb3c049be1cc007123647a976c67af0902b366aff13acf3da
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000,1.69,13047,United Kingdom,11,Online Retail.xlsx,Online Retail,2026-08-09T20:24:05.199Z,75454c4ed0bc3e1e48f278af66b0f1de1a082a6ed8765e348cd8bc5f593895c6


## 8. Design deterministic Lab 4 source batches

The batches below are generated from stable source-row hashing. Re-running this notebook therefore selects the same base rows and overwrites only dedicated Lab 4 staging paths.

| Batch | Purpose | Expected shape |
|---|---|---|
| `initial` | First Bronze load | About 80% of source rows, original schema plus metadata |
| `incremental` | Rerun/incremental test | About 20%, same schema as initial |
| `evolved` | Controlled source evolution | Adds `SalesChannel` and `PromotionCode` |
| `invalid` | Silver quality/quarantine test | Deliberate negative price, blank country, and missing description |
| `schema_mismatch` | Enforcement failure test | Changes `Quantity` from integer to string |
| `schema_evolution` | Approved evolution test | Same approved new columns as evolved |
| `scd_changes` | Product dimension changes | Deterministic description and price updates with effective timestamp |

> These synthetic variations are isolated from the baseline initial and incremental batches. This prevents one experiment from contaminating another.

In [0]:
split_df = prepared_df.withColumn(
    "_split_bucket",
    F.pmod(F.xxhash64(F.col("_source_row_number")), F.lit(10)),
)

initial_df = split_df.filter(F.col("_split_bucket") < 8).drop("_split_bucket")
incremental_df = split_df.filter(F.col("_split_bucket") >= 8).drop("_split_bucket")

evolution_seed_df = incremental_df.orderBy("_source_row_number").limit(5000)
evolved_df = (
    evolution_seed_df
    .withColumn("SalesChannel", F.lit("online"))
    .withColumn(
        "PromotionCode",
        F.when(
            F.pmod(F.xxhash64(F.col("_source_row_number")), F.lit(2)) == 0,
            F.lit("WELCOME10"),
        ).otherwise(F.lit(None).cast("string")),
    )
)

invalid_df = (
    incremental_df.orderBy("_source_row_number").limit(500)
    .withColumn("Description", F.lit(None).cast("string"))
    .withColumn("UnitPrice", F.lit(-1.0))
    .withColumn("Country", F.lit(""))
    .withColumn("_test_scenario", F.lit("deliberate_quality_violation"))
)

schema_mismatch_df = (
    incremental_df.orderBy("_source_row_number").limit(1000)
    .withColumn("Quantity", F.col("Quantity").cast("string"))
)
schema_evolution_df = evolved_df

product_window = Window.partitionBy("StockCode").orderBy(
    F.col("InvoiceDate").desc_nulls_last(),
    F.col("Description").asc_nulls_last(),
)
product_base_df = (
    prepared_df
    .filter(F.col("StockCode").isNotNull())
    .withColumn("_product_rank", F.row_number().over(product_window))
    .filter(F.col("_product_rank") == 1)
    .select("StockCode", "Description", "UnitPrice")
)
scd_changes_df = (
    product_base_df.orderBy("StockCode").limit(500)
    .withColumn(
        "Description",
        F.concat(F.coalesce(F.col("Description"), F.lit("Unknown product")), F.lit(" - UPDATED")),
    )
    .withColumn("UnitPrice", F.round(F.coalesce(F.col("UnitPrice"), F.lit(0.0)) * F.lit(1.05), 2))
    .withColumn("change_effective_at", F.to_timestamp(F.lit("2012-01-01 00:00:00")))
    .withColumn("change_reason", F.lit("Lab 4 controlled product update"))
)

## 9. Write the isolated batches

Each output uses `overwrite` inside a dedicated staging or test path. This makes source preparation idempotent: running it twice replaces the same generated test data instead of multiplying files.

Nothing is copied into `landing` yet. The next notebook controls that promotion so Bronze ingestion evidence starts from a known state.

In [0]:
batch_outputs = {
    "initial": (initial_df, paths["staging_initial"], 16),
    "incremental": (incremental_df, paths["staging_incremental"], 4),
    "evolved": (evolved_df, paths["staging_evolved"], 2),
    "invalid": (invalid_df, paths["staging_invalid"], 1),
    "schema_mismatch": (schema_mismatch_df, paths["schema_mismatch"], 1),
    "schema_evolution": (schema_evolution_df, paths["schema_evolution"], 2),
    "scd_changes": (scd_changes_df, paths["scd_changes"], 1),
}

for batch_name, (batch_df, output_path, partition_count) in batch_outputs.items():
    (
        batch_df
        .repartition(partition_count)
        .write
        .mode("overwrite")
        .parquet(output_path)
    )
    print(f"Wrote {batch_name}: {output_path}")

Wrote initial: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial
Wrote incremental: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/incremental
Wrote evolved: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/evolved
Wrote invalid: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/invalid
Wrote schema_mismatch: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/schema_mismatch
Wrote schema_evolution: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/schema_evolution
Wrote scd_changes: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/scd_changes


## 10. Validate completeness, isolation, and schema scenarios

The following assertions are part of the notebook, not optional debugging output. They prove that:

1. initial plus incremental rows reproduce the complete source;
2. the two baseline batches do not overlap by source row;
3. the evolved batch really contains the approved new columns;
4. the mismatch batch really changed `Quantity` to string;
5. the SCD change set is non-empty.

In [0]:
initial_count = initial_df.count()
incremental_count = incremental_df.count()

if initial_count + incremental_count != prepared_row_count:
    raise AssertionError("Initial and incremental counts do not reproduce the source count.")

overlap_count = (
    initial_df.select("_source_row_number")
    .join(
        incremental_df.select("_source_row_number"),
        on="_source_row_number",
        how="inner",
    )
    .count()
)
if overlap_count != 0:
    raise AssertionError(f"Initial/incremental overlap detected: {overlap_count} rows")

if not {"SalesChannel", "PromotionCode"}.issubset(set(evolved_df.columns)):
    raise AssertionError("The evolved batch does not contain the expected new columns.")

quantity_type = dict(schema_mismatch_df.dtypes)["Quantity"]
if quantity_type != "string":
    raise AssertionError(f"Expected Quantity to be string, found {quantity_type}")

if scd_changes_df.limit(1).count() == 0:
    raise AssertionError("The SCD change batch is empty.")

print("Core source-preparation assertions passed.")

Core source-preparation assertions passed.


In [0]:
summary_rows = []

for batch_name, (_, output_path, _) in batch_outputs.items():
    persisted_df = spark.read.parquet(output_path)
    parquet_file_count = sum(
        1 for item in dbutils.fs.ls(output_path)
        if item.name.endswith(".parquet")
    )
    summary_rows.append(
        {
            "batch_name": batch_name,
            "row_count": persisted_df.count(),
            "parquet_files": parquet_file_count,
            "column_count": len(persisted_df.columns),
            "output_path": output_path,
        }
    )

batch_summary_df = spark.createDataFrame(summary_rows).orderBy("batch_name")
display(batch_summary_df)

batch_name,column_count,output_path,parquet_files,row_count
evolved,15,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/evolved,2,5000
incremental,13,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/incremental,4,108172
initial,13,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial,16,433737
invalid,14,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/invalid,1,500
scd_changes,5,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/scd_changes,1,500
schema_evolution,15,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/schema_evolution,2,5000
schema_mismatch,13,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/schema_mismatch,1,1000


## 11. Evidence to capture

For the Lab 4 README, capture these two outputs after a successful run:

1. the raw source profile showing row count, duplicates, and null counts;
2. the final batch summary showing row counts, file counts, column counts, and isolated paths.

A useful filename convention is:

- `01_source_profile.png`
- `02_source_batch_summary.png`

In [0]:
print("Lab 4 source preparation completed successfully.")
print(f"Source rows: {prepared_row_count:,}")
print(f"Initial rows: {initial_count:,}")
print(f"Incremental rows: {incremental_count:,}")
print(f"Landing remains unchanged: {landing_path}")
print("The generated scenarios are ready for Bronze ingestion and later schema/SCD tests.")

Lab 4 source preparation completed successfully.
Source rows: 541,909
Initial rows: 433,737
Incremental rows: 108,172
Landing remains unchanged: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing
The generated scenarios are ready for Bronze ingestion and later schema/SCD tests.


## Next notebook

Continue with **`lab04_02_bronze_ingestion.ipynb`**.

That notebook will:

- reset the Lab 4 landing area only when explicitly requested;
- promote the `initial` batch into landing;
- ingest the files into the Bronze Delta table with an explicit schema;
- keep schema-tracking and checkpoint paths separate;
- preserve file and ingestion metadata;
- rerun with the same checkpoint to demonstrate incremental, idempotent behavior.